## Exploring predictions

In [1]:
import os
import sys
import importlib as imp
import numpy as np
import torch

from utils import utils
from data_builder import build_tags
from data_builder import data_loader
from predictor import inference

from data_builder import read_landsat

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"pytorch version = {torch.__version__}")

python version = 3.12.0 | packaged by conda-forge | (main, Oct  3 2023, 08:36:57) [Clang 15.0.7 ]
numpy version = 1.26.4
pytorch version = 2.1.2.post3


In [ ]:
# GET config
EXP_NAME = "exp_001"
REWRITE = True

config = utils.get_config(EXP_NAME)
config["mode"] = "inference"
config["trainer"]["batch_size"] = config["inference"]["batch_size"]

directory_paths = utils.get_directories()
SAVE_MODEL_DIR = directory_paths["save_model_dir"]
DATA_DIR = directory_paths["data_dir"]
FIGURE_DIR = directory_paths["figures_dir"]
PREDICTIONS_DIR = directory_paths["predictions_dir"]
LANDSAT_DIR = directory_paths["landsat_dir"]
MOSAICS_DIR = directory_paths["mosaics_dir"]

In [ ]:
# GET THE DATA
config["data"]["inference_region"] = (5.0, 8.0, 2.0, 6.0)

# set and get important config
(lat_s_bound, lat_n_bound, lon_w_bound, lon_e_bound) = read_landsat.get_landsat_bounds(
    config, region=config["data"]["inference_region"]
)

# load the model
model = utils.load_model(config, clean=False)

for year in config["data"]["inference_years"]:
    print(" --- " + str(year) + "---")
    config["data"]["inference_years"] = (year,)
    filenames_list = []

    for latfile in np.arange(
        lat_s_bound + config["tile_len_deg"],
        lat_n_bound + config["tile_len_deg"],
        config["tile_len_deg"],
    ):
        for lonfile in np.arange(lon_w_bound, lon_e_bound, config["tile_len_deg"]):
            # check if landsat tile exists, if so, get it.
            config["tile"] = (
                latfile - config["tile_len_deg"],
                latfile,
                lonfile,
                lonfile + config["tile_len_deg"],
            )
            landsat_file = read_landsat.get_input_filename(
                config["data"]["inference_years"], (latfile,), (lonfile,), config
            )

            if os.path.isfile(LANDSAT_DIR + landsat_file[0] + ".tif") is False:
                continue

            # TODO: check if landsat tile is all water, if so, create prediction file of all NODATA

            # check if prediction file already exists
            predictions_filename = utils.get_predictions_filename(
                config, landsat_file[0]
            )
            filenames_list.append(predictions_filename)
            if (
                os.path.isfile(predictions_filename)
                and REWRITE is False
            ):
                continue
            print(landsat_file[0])

            # GET THE SAMPLE TAGS
            tags, __ = build_tags.get_tags(config)
            if len(tags[0]) == 0:
                continue

            # MAKE PREDICTIONS and SAVE AS TIFF
            ds_inf = data_loader.CustomData(config, tags)
            inf_loader = torch.utils.data.DataLoader(
                ds_inf,
                batch_size=None,
                batch_sampler=None,
                shuffle=False,
                drop_last=False,
                pin_memory=config["inference"]["pin_memory"],
                num_workers=config["inference"]["num_workers"],
            )
            hfi_predict, hfi_labels, latlon_bounds = inference.make_predictions(
                config, model, tags, inf_loader
            )

            meta_data = inference.save_predictions_tif(
                hfi_predict,
                predictions_filename,
                latlon_bounds=latlon_bounds,
            )

            filenames_list.append(predictions_filename)
            print("\n")

    # TILE THE PREDICTIONS TOGETHER
    mosaic_filename = (
        MOSAICS_DIR
        + config["exp_name"]
        + "_"
        + str(config["data"]["inference_years"][0])
        + "_mlhfi_mosaic.tif"
    )
    mosaic, mosaic_trans = inference.create_mosaic(filenames_list)
    meta_data = inference.save_predictions_tif(
        mosaic, mosaic_filename, trans=mosaic_trans
    )
    print("mosaic saved.")

loading model from:  saved/models//exp_001/exp_001_seed22/exp_001_seed22.pt
 --- 2023---
landsat_10lat_0lon_2023
output region shape = (371, 371)
n_inference = (137641,)
batch 0 of 135 - 2.637s - 0.0025752s/sample
batch 100 of 135 - 18.454s - 0.0001784s/sample

Execution time: 33.757s
Number samples: 137641
Time per sample: 0.0002453s


mosaic saved.
